In [45]:
import ee
import pandas as pd
import numpy as np

In [2]:
ee.Initialize(project="sundarbans-climate-analysis")

In [3]:
print("Google Earth Engine initialized successfully!")

Google Earth Engine initialized successfully!


In [4]:
districts = ee.FeatureCollection(
    "FAO/GAUL/2015/level2"
)

C:\Users\Projit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level2

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level2

  warnings.warn(warning, category=DeprecationWarning)


In [5]:
bangladesh_districts = districts.filter(
    ee.Filter.eq(
        "ADM0_NAME",
        "Bangladesh"
    )
)

In [6]:
target_districts = bangladesh_districts.filter(
    ee.Filter.inList(
        "ADM2_NAME",
        [
            "Satkhira",
            "Khulna",
            "Bagerhat"
        ]
    )
)

In [7]:
print(
    "Selected districts:"
)

Selected districts:


In [8]:
print(
    target_districts
    .aggregate_array("ADM2_NAME")
    .getInfo()
)

['Bagerhat', 'Khulna', 'Satkhira']


In [9]:
study_area = target_districts.geometry()

print(
    "Study area created successfully."
)

Study area created successfully.


In [10]:
era5 = ee.ImageCollection(
    "ECMWF/ERA5_LAND/MONTHLY_AGGR"
)

In [11]:
rainfall_collection = (
    era5
    .filterDate(
        "2000-01-01",
        "2026-01-01"
    )
    .select(
        "total_precipitation_sum"
    )
)

In [12]:
print(
    "Number of monthly images:",
    rainfall_collection.size().getInfo()
)

Number of monthly images: 312


In [13]:
start_date = ee.Date(
    "2000-01-01"
)

In [14]:
end_date = ee.Date(
    "2026-01-01"
)

In [15]:
number_of_months = (
    end_date
    .difference(
        start_date,
        "month"
    )
)

In [16]:
month_sequence = ee.List.sequence(
    0,
    number_of_months.subtract(1)
)

In [17]:
def calculate_monthly_rainfall(month_number):

    # Current month
    current_date = start_date.advance(
        month_number,
        "month"
    )

    # Next month
    next_date = current_date.advance(
        1,
        "month"
    )


    # Get image for the current month

    monthly_image = (
        rainfall_collection
        .filterDate(
            current_date,
            next_date
        )
        .first()
    )

    rainfall_meters = (
        monthly_image
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=11132,
            maxPixels=1000000
        )
        .get(
            "total_precipitation_sum"
        )
    )

    rainfall_mm = (
        ee.Number(rainfall_meters)
        .multiply(1000)
    )


    # Create feature

    return ee.Feature(
        None,
        {
            "Year":
                current_date.get("year"),

            "Month_Number":
                current_date.get("month"),

            "Month":
                current_date.format("MMMM"),

            "Date":
                current_date.format(
                    "YYYY-MM-dd"
                ),

            "Rainfall_mm":
                rainfall_mm
        }
    )


In [18]:
monthly_rainfall = (
    ee.FeatureCollection(
        month_sequence.map(
            calculate_monthly_rainfall
        )
    )
)

In [19]:
print(
    "Monthly rainfall calculation completed."
)


Monthly rainfall calculation completed.


In [20]:
print(
    "Number of records:",
    monthly_rainfall.size().getInfo()
)

Number of records: 312


In [21]:
data = monthly_rainfall.getInfo()

rows = []

for feature in data["features"]:

    properties = feature["properties"]

    rows.append(
        {
            "Year":
                properties.get("Year"),

            "Month_Number":
                properties.get(
                    "Month_Number"
                ),

            "Month":
                properties.get("Month"),

            "Date":
                properties.get("Date"),

            "Rainfall_mm":
                properties.get(
                    "Rainfall_mm"
                )
        }
    )


df_rainfall = pd.DataFrame(
    rows
)

In [22]:
df_rainfall = (
    df_rainfall
    .sort_values(
        by="Date"
    )
    .reset_index(
        drop=True
    )
)

In [23]:
df_rainfall[
    "Rainfall_mm"
] = (
    df_rainfall[
        "Rainfall_mm"
    ]
    .round(2)
)

In [24]:
print(
    df_rainfall.head(12)
)

print(
    "\nTotal records:",
    len(df_rainfall)
)

    Year  Month_Number      Month        Date  Rainfall_mm
0   2000             1    January  2000-01-01         7.85
1   2000             2   February  2000-02-01        24.16
2   2000             3      March  2000-03-01        11.18
3   2000             4      April  2000-04-01        59.38
4   2000             5        May  2000-05-01       198.05
5   2000             6       June  2000-06-01       334.97
6   2000             7       July  2000-07-01       339.04
7   2000             8     August  2000-08-01       285.75
8   2000             9  September  2000-09-01       252.15
9   2000            10    October  2000-10-01       179.99
10  2000            11   November  2000-11-01         6.13
11  2000            12   December  2000-12-01         0.03

Total records: 312


In [25]:
output_file = (
    "Sundarbans_Monthly_Rainfall_2000_2025.xlsx"
)

df_rainfall.to_excel(
    output_file,
    index=False
)


print(
    "\nExcel file created successfully:"
)

print(
    output_file
)


Excel file created successfully:
Sundarbans_Monthly_Rainfall_2000_2025.xlsx


In [26]:
print(
    "\nDate range:"
)

print(
    df_rainfall["Date"].min(),
    "to",
    df_rainfall["Date"].max()
)


print(
    "\nMissing values:"
)

print(
    df_rainfall.isnull().sum()
)


print(
    "\nLast 12 months:"
)

print(
    df_rainfall.tail(12)
)


Date range:
2000-01-01 to 2025-12-01

Missing values:
Year            0
Month_Number    0
Month           0
Date            0
Rainfall_mm     0
dtype: int64

Last 12 months:
     Year  Month_Number      Month        Date  Rainfall_mm
300  2025             1    January  2025-01-01         1.61
301  2025             2   February  2025-02-01        12.29
302  2025             3      March  2025-03-01         6.16
303  2025             4      April  2025-04-01       104.12
304  2025             5        May  2025-05-01       209.89
305  2025             6       June  2025-06-01       387.29
306  2025             7       July  2025-07-01       614.95
307  2025             8     August  2025-08-01       323.58
308  2025             9  September  2025-09-01       265.17
309  2025            10    October  2025-10-01       148.95
310  2025            11   November  2025-11-01         5.48
311  2025            12   December  2025-12-01         0.25


In [ ]:
## EXPLORATORY DATA ANALYSIS (EDA), Rainfall

In [27]:
file_path = "Sundarbans_Monthly_Rainfall_2000_2025.xlsx"

In [28]:
df = pd.read_excel(file_path)

In [35]:
print (df.head())

   Year  Month_Number     Month       Date  Rainfall_mm
0  2000             1   January 2000-01-01         7.85
1  2000             2  February 2000-02-01        24.16
2  2000             3     March 2000-03-01        11.18
3  2000             4     April 2000-04-01        59.38
4  2000             5       May 2000-05-01       198.05


In [29]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


In [30]:
df["Rainfall_mm"] = pd.to_numeric(
    df["Rainfall_mm"], errors="coerce"
)

In [36]:
df = df.sort_values("Date").reset_index(drop=True)

In [37]:
rainfall = df["Rainfall_mm"].dropna()

In [ ]:
# 2. DATASET OVERVIEW

In [38]:
overview = pd.DataFrame({
    "Metric": [
        "Total Observations",
        "Total Variables",
        "Start Date",
        "End Date",
        "Missing Dates",
        "Missing Rainfall Values",
        "Duplicate Records",
        "Negative Rainfall Values",
        "Zero Rainfall Values"
    ],

    "Value": [
        len(df),
        len(df.columns),
        df["Date"].min(),
        df["Date"].max(),
        df["Date"].isna().sum(),
        df["Rainfall_mm"].isna().sum(),
        df.duplicated().sum(),
        (df["Rainfall_mm"] < 0).sum(),
        (df["Rainfall_mm"] == 0).sum()
    ]
})

In [ ]:
# 3. CENTRAL TENDENCY

In [41]:
mean_value = rainfall.mean()
median_value = rainfall.median()
mode_value = rainfall.mode().tolist()

central_tendency = pd.DataFrame({
    "Metric": ["Mean", "Median", "Mode"],
    "Value": [
        mean_value,
        median_value,
        str(mode_value)
    ]
})


In [ ]:
# 4. DISPERSION ANALYSIS

In [42]:
q1 = rainfall.quantile(0.25)
q2 = rainfall.quantile(0.50)
q3 = rainfall.quantile(0.75)

iqr = q3 - q1

dispersion = pd.DataFrame({
    "Metric": [
        "Minimum",
        "Maximum",
        "Range",
        "Variance",
        "Standard Deviation",
        "Q1 (25%)",
        "Q2 (50%)",
        "Q3 (75%)",
        "Interquartile Range",
        "Coefficient of Variation (%)"
    ],

    "Value": [
        rainfall.min(),
        rainfall.max(),
        rainfall.max() - rainfall.min(),
        rainfall.var(),
        rainfall.std(),
        q1,
        q2,
        q3,
        iqr,
        (rainfall.std() / rainfall.mean()) * 100
        if rainfall.mean() != 0 else np.nan
    ]
})

In [ ]:
# 5. DISTRIBUTION SHAPE

In [43]:
distribution = pd.DataFrame({
    "Metric": [
        "Skewness",
        "Excess Kurtosis"
    ],

    "Value": [
        rainfall.skew(),
        rainfall.kurt()
    ]
})

In [ ]:
# 6. OUTLIER DETECTION USING IQR

In [46]:
lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)

outliers = df[
    (df["Rainfall_mm"] < lower_bound) |
    (df["Rainfall_mm"] > upper_bound)
].copy()

outliers["Outlier_Type"] = np.where(
    outliers["Rainfall_mm"] < lower_bound,
    "Below Lower Bound",
    "Above Upper Bound"
)

outlier_summary = pd.DataFrame({
    "Metric": [
        "Q1",
        "Q3",
        "IQR",
        "Lower Bound",
        "Upper Bound",
        "Potential Outliers",
        "Outlier Percentage (%)"
    ],

    "Value": [
        q1,
        q3,
        iqr,
        lower_bound,
        upper_bound,
        len(outliers),
        (len(outliers) / len(rainfall)) * 100
        if len(rainfall) else np.nan
    ]
})

In [ ]:
# 7. MONTHLY STATISTICAL SUMMARY

In [47]:
df["Year"] = df["Date"].dt.year
df["Month_Number"] = df["Date"].dt.month
df["Month"] = df["Date"].dt.month_name()

monthly_summary = (
    df.groupby(["Month_Number", "Month"])["Rainfall_mm"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Standard_Deviation="std",
        Minimum="min",
        Maximum="max"
    )
    .reset_index()
    .sort_values("Month_Number")
)

In [48]:
# 8. ANNUAL STATISTICAL SUMMARY

In [49]:
annual_summary = (
    df.groupby("Year")["Rainfall_mm"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Standard_Deviation="std",
        Minimum="min",
        Maximum="max",
        Total_Rainfall="sum"
    )
    .reset_index()
)


In [50]:
# 9. EXPORT RESULTS TO EXCEL

In [52]:
import os

In [53]:
os.makedirs("EDA", exist_ok=True)

In [56]:
output_file = os.path.join("EDA", "Rainfall_EDA_2_1_Descriptive_Statistics.xlsx")


with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    overview.to_excel(
        writer,
        sheet_name="Dataset_Overview",
        index=False
    )

    central_tendency.to_excel(
        writer,
        sheet_name="Central_Tendency",
        index=False
    )

    dispersion.to_excel(
        writer,
        sheet_name="Dispersion",
        index=False
    )

    distribution.to_excel(
        writer,
        sheet_name="Distribution_Shape",
        index=False
    )

    outlier_summary.to_excel(
        writer,
        sheet_name="Outlier_Summary",
        index=False
    )

    outliers.to_excel(
        writer,
        sheet_name="Potential_Outliers",
        index=False
    )

    monthly_summary.to_excel(
        writer,
        sheet_name="Monthly_Summary",
        index=False
    )

    annual_summary.to_excel(
        writer,
        sheet_name="Annual_Summary",
        index=False
    )



In [57]:
# 10. DISPLAY RESULTS

In [58]:
print("\nDataset Overview:")
print(overview.to_string(index=False))


Dataset Overview:
                  Metric               Value
      Total Observations                 312
         Total Variables                   5
              Start Date 2000-01-01 00:00:00
                End Date 2025-12-01 00:00:00
           Missing Dates                   0
 Missing Rainfall Values                   0
       Duplicate Records                   0
Negative Rainfall Values                   0
    Zero Rainfall Values                   0


In [59]:
print("\nCentral Tendency:")
print(central_tendency.to_string(index=False))


Central Tendency:
Metric              Value
  Mean         156.259103
Median            104.835
  Mode [0.13, 0.25, 0.44]


In [61]:
print("\nDispersion Analysis:")
print(dispersion.to_string(index=False))


Dispersion Analysis:
                      Metric        Value
                     Minimum     0.020000
                     Maximum   670.210000
                       Range   670.190000
                    Variance 24080.368894
          Standard Deviation   155.178507
                    Q1 (25%)    12.855000
                    Q2 (50%)   104.835000
                    Q3 (75%)   275.430000
         Interquartile Range   262.575000
Coefficient of Variation (%)    99.308459


In [62]:
print("\nDistribution Shape:")
print(distribution.to_string(index=False))


Distribution Shape:
         Metric     Value
       Skewness  0.807591
Excess Kurtosis -0.209025


In [63]:
print("\nOutlier Summary:")
print(outlier_summary.to_string(index=False))


Outlier Summary:
                Metric       Value
                    Q1   12.855000
                    Q3  275.430000
                   IQR  262.575000
           Lower Bound -381.007500
           Upper Bound  669.292500
    Potential Outliers    1.000000
Outlier Percentage (%)    0.320513


In [64]:
print("\nMonthly Summary:")
print(monthly_summary.to_string(index=False))


Monthly Summary:
 Month_Number     Month  Count       Mean  Median  Standard_Deviation  Minimum  Maximum
            1   January     26   8.957308   1.820           11.934061     0.18    42.13
            2  February     26  15.739615   7.100           19.236940     0.36    78.66
            3     March     26  32.821538  24.430           30.599405     3.38   118.94
            4     April     26  80.127692  66.425           66.170683     8.45   242.38
            5       May     26 177.279231 153.955           99.778371    50.81   383.20
            6      June     26 319.567308 330.130          110.248927   146.08   497.85
            7      July     26 387.246538 344.650          122.934959   212.78   670.21
            8    August     26 350.103462 322.300           92.335638   195.69   592.11
            9 September     26 271.843077 273.645           47.825590   196.53   380.26
           10   October     26 174.164231 163.415           72.836964    62.17   390.19
           11 

In [65]:
print("\nAnnual Summary:")
print(annual_summary.to_string(index=False))


Annual Summary:
 Year  Count       Mean  Median  Standard_Deviation  Minimum  Maximum  Total_Rainfall
 2000     12 141.556667 119.685          137.470762     0.03   339.04         1698.68
 2001     12 177.050000 142.370          166.164517     0.04   457.42         2124.60
 2002     12 170.587500 105.620          162.013837     0.70   497.85         2047.05
 2003     12 139.342500  82.340          140.183808     0.94   432.58         1672.11
 2004     12 158.290833 128.280          158.138966     0.36   380.85         1899.49
 2005     12 166.521667  93.905          171.204391     0.95   499.62         1998.26
 2006     12 145.656667  50.490          166.839722     0.05   444.44         1747.88
 2007     12 200.685000 147.920          190.975328     0.02   610.75         2408.22
 2008     12 170.484167  82.030          186.108042     0.13   545.62         2045.81
 2009     12 134.419167  73.705          150.217725     0.08   421.55         1613.03
 2010     12 119.210000  86.205      

In [66]:
print("\nAnalysis completed successfully!")
print("Output saved:", output_file)


Analysis completed successfully!
Output saved: EDA\Rainfall_EDA_2_1_Descriptive_Statistics.xlsx
